# 3. COMPASS Geneset-Level Bottleneck Embedding Extraction

Извлечение geneset-level (до concept bottleneck) эмбеддингов из COMPASS foundation model.

**Архитектура COMPASS:**
```
Gene TPM → TransformerEncoder → GenesetProjector (133 × 32) → CellPathwayProjector (44)
                                 ↑ WE EXTRACT HERE            ↑ concept bottleneck
```

Модель: `finetuner_pft_all.pt` — fine-tuned PFT на всех 33 TCGA solid cancer types.

## 0. Environment Setup

In [ ]:
# !rm -rf batchcor-rna-embeds/  # if already cloned
!git clone https://github.com/onion-42/batchcor-rna-embeds.git
# or: !cd batchcor-rna-embeds && git pull origin main

%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel

In [ ]:
import os
import sys
import shutil
from pathlib import Path

import numpy as np
import torch
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL,
    DIAGNOSIS_COL,
    COMPASS_MODEL_URL,
    COMPASS_EMBEDDING_KEY,
    COMPASS_PCA_KEY,
    COMPASS_UMAP_KEY,
    COMPASS_METADATA_KEY,
    COHORT_CANCER_CODE,
    SEED,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.compass_embedder import run_compass_pipeline

set_logger(level="INFO")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: {}", DEVICE)
logger.info("PyTorch: {}, CUDA available: {}", torch.__version__, torch.cuda.is_available())

## 1. Mount Drive & Define Paths

**ВАЖНО:** Результаты сохраняются в `data/interim`, а НЕ в `data/raw`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Укажите путь к данным на Google Drive
DATA_RAW_DIR = Path('/content/drive/MyDrive/data/raw')
DATA_INTERIM_DIR = Path('/content/drive/MyDrive/data/interim')
DATA_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path('/content/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

logger.info("DATA_RAW_DIR: {}", DATA_RAW_DIR)
logger.info("DATA_INTERIM_DIR: {}", DATA_INTERIM_DIR)
logger.info("Available cohorts (raw): {}", [p.name for p in sorted(DATA_RAW_DIR.glob('*.adata.zarr'))])

## 2. Download COMPASS Model

In [ ]:
MODEL_PATH = MODEL_DIR / "finetuner_pft_all.pt"

if not MODEL_PATH.exists():
    logger.info("Downloading COMPASS model from {}", COMPASS_MODEL_URL)
    !wget -q -O {MODEL_PATH} {COMPASS_MODEL_URL}
    logger.info("Model downloaded: {} ({:.1f} MB)", MODEL_PATH, MODEL_PATH.stat().st_size / 1e6)
else:
    logger.info("Model already exists: {}", MODEL_PATH)

## 3a. Copy Already-Processed Cohorts from raw → interim

Если когорты уже были обработаны и сохранены в `raw` (с эмбеддингами), копируем их в `interim`.

In [ ]:
already_processed = [
    "NSCLC_Tissue_ICI_Pred",
    "PUB_ccRCC_Immotion150_and_151_ICI",
    "PUB_ccRCC_Immotion150_and_151_TKI",
]

for name in already_processed:
    src = DATA_RAW_DIR / f"{name}.adata.zarr"
    dst = DATA_INTERIM_DIR / f"{name}.adata.zarr"

    if not src.exists():
        logger.warning("Source not found, skipping: {}", src)
        continue

    if dst.exists():
        logger.info("Already in interim, skipping: {}", dst.name)
        continue

    logger.info("Copying {} → {}", src.name, DATA_INTERIM_DIR)
    shutil.copytree(str(src), str(dst))
    logger.info("Done: {}", dst.name)

logger.info("\nInterim contents: {}", [p.name for p in sorted(DATA_INTERIM_DIR.glob('*.adata.zarr'))])

## 3b. Process Remaining Cohorts

Для каждой когорты:
1. Load AnnData from raw Zarr
2. Extract COMPASS geneset-level embeddings (132 × 32 = 4,224-dim)
3. Compute PCA-128D + UMAP-3D
4. Store metadata in `.uns`
5. Save to `data/interim`

In [ ]:
BATCH_SIZE = 128

for cohort_name, cancer_code in COHORT_CANCER_CODE.items():
    zarr_path = DATA_RAW_DIR / f"{cohort_name}.adata.zarr"
    interim_path = DATA_INTERIM_DIR / f"{cohort_name}.adata.zarr"

    if not zarr_path.exists():
        logger.warning("Cohort not found in raw, skipping: {}", zarr_path)
        continue

    if interim_path.exists():
        logger.info("Already in interim, skipping: {}", cohort_name)
        continue

    logger.info("\n{'=' * 70}")
    logger.info("Processing cohort: {}", cohort_name)
    logger.info("{'=' * 70}")

    # Load from raw
    adata = load_cohort(zarr_path)
    logger.info("Loaded: {} samples × {} genes", adata.n_obs, adata.n_vars)
    logger.info("Batch distribution:\n{}", adata.obs[BATCH_COL].value_counts())
    logger.info("Diagnosis distribution:\n{}", adata.obs[DIAGNOSIS_COL].value_counts())

    # Run COMPASS pipeline
    adata = run_compass_pipeline(
        adata,
        model_path=MODEL_PATH,
        cancer_code=cancer_code,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        seed=SEED,
    )

    # Verify
    logger.info("Verification:")
    logger.info("  .obsm['{}'] shape: {}, dtype: {}",
        COMPASS_EMBEDDING_KEY,
        adata.obsm[COMPASS_EMBEDDING_KEY].shape,
        adata.obsm[COMPASS_EMBEDDING_KEY].dtype,
    )
    logger.info("  .obsm['{}'] shape: {}",
        COMPASS_PCA_KEY, adata.obsm[COMPASS_PCA_KEY].shape,
    )
    logger.info("  .obsm['{}'] shape: {}",
        COMPASS_UMAP_KEY, adata.obsm[COMPASS_UMAP_KEY].shape,
    )
    logger.info("  .uns['{}']: {}",
        COMPASS_METADATA_KEY, adata.uns[COMPASS_METADATA_KEY],
    )

    # Save to INTERIM (not raw!)
    save_adata_zarr(adata, interim_path)
    logger.info("Saved to interim: {}", interim_path)

    # Free memory
    del adata
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

logger.info("\nAll cohorts processed successfully.")
logger.info("Interim contents: {}", [p.name for p in sorted(DATA_INTERIM_DIR.glob('*.adata.zarr'))])

## 4. Verification — Single Cohort Spot Check

In [ ]:
# Quick reload from interim and verify
test_cohort = "NSCLC_Tissue_ICI_Pred"
test_path = DATA_INTERIM_DIR / f"{test_cohort}.adata.zarr"

if test_path.exists():
    adata_check = load_cohort(test_path)
    
    assert COMPASS_EMBEDDING_KEY in adata_check.obsm, "Missing embedding key!"
    assert COMPASS_PCA_KEY in adata_check.obsm, "Missing PCA key!"
    assert COMPASS_UMAP_KEY in adata_check.obsm, "Missing UMAP key!"
    assert COMPASS_METADATA_KEY in adata_check.uns, "Missing metadata key!"
    
    emb = adata_check.obsm[COMPASS_EMBEDDING_KEY]
    assert emb.dtype == np.float32, f"Wrong dtype: {emb.dtype}"
    assert emb.shape[0] == adata_check.n_obs, "Row count mismatch!"
    
    logger.info("All assertions passed for '{}'.", test_cohort)
    logger.info("  Embedding shape: {}", emb.shape)
    logger.info("  PCA shape: {}", adata_check.obsm[COMPASS_PCA_KEY].shape)
    logger.info("  UMAP shape: {}", adata_check.obsm[COMPASS_UMAP_KEY].shape)
    logger.info("  Metadata: {}", adata_check.uns[COMPASS_METADATA_KEY])
else:
    logger.warning("Test cohort not found in interim: {}", test_path)